In [6]:
# =====================================================================
# CELL 0: KHỞI TẠO DASHBOARD REPORT MANAGER
# Tự động tạo và append nội dung vào file IMP302_Report.html
# =====================================================================

import base64
from io import BytesIO
import json
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np

REPORT_FILE = "IMP302_Report.html"


def fig_to_base64(fig):
  """Chuyển matplotlib figure thành chuỗi Base64 nhúng thẳng vào HTML."""
  buf = BytesIO()
  fig.savefig(buf, format="png", bbox_inches="tight", dpi=130)
  buf.seek(0)
  img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
  plt.close(fig)
  return f"data:image/png;base64,{img_b64}"


def init_dashboard():
  html_template = """<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>IMP302 - Image Processing Dashboard</title>
    <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600&display=swap" rel="stylesheet">
    <style>
        :root {
            --bg-body: #0b0f19;
            --bg-card: rgba(23, 32, 54, 0.7);
            --border-color: rgba(255, 255, 255, 0.08);
            --accent-cyan: #06b6d4;
            --accent-blue: #3b82f6;
            --accent-orange: #f97316;
            --accent-green: #10b981;
            --text-main: #f8fafc;
            --text-sub: #94a3b8;
        }
        * { box-sizing: border-box; margin: 0; padding: 0; }
        body {
            font-family: 'Plus Jakarta Sans', sans-serif;
            background-color: var(--bg-body);
            color: var(--text-main);
            line-height: 1.6;
            padding: 30px 20px;
        }
        .container { max-width: 1200px; margin: 0 auto; }
        header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            border-bottom: 1px solid var(--border-color);
            padding-bottom: 20px;
            margin-bottom: 30px;
        }
        .logo-title { font-size: 1.5rem; font-weight: 700; color: #fff; }
        .logo-title span { color: var(--accent-cyan); }
        .badge {
            background: rgba(6, 182, 212, 0.15);
            color: var(--accent-cyan);
            border: 1px solid var(--accent-cyan);
            padding: 5px 12px;
            border-radius: 20px;
            font-size: 0.8rem;
            font-family: 'JetBrains Mono', monospace;
        }
        .card {
            background: var(--bg-card);
            backdrop-filter: blur(12px);
            border: 1px solid var(--border-color);
            border-radius: 16px;
            padding: 25px;
            margin-bottom: 30px;
            box-shadow: 0 10px 30px -10px rgba(0,0,0,0.5);
            transition: border-color 0.2s;
        }
        .card:hover { border-color: rgba(6, 182, 212, 0.3); }
        .card-header {
            display: flex;
            justify-content: space-between;
            align-items: center;
            margin-bottom: 15px;
            border-bottom: 1px solid var(--border-color);
            padding-bottom: 10px;
        }
        .card-header h2 { font-size: 1.25rem; font-weight: 600; color: #fff; }
        .formula-box {
            background: rgba(0, 0, 0, 0.3);
            border-left: 3px solid var(--accent-blue);
            padding: 10px 15px;
            font-family: 'JetBrains Mono', monospace;
            font-size: 0.9rem;
            color: #cbd5e1;
            margin-bottom: 20px;
            border-radius: 0 8px 8px 0;
        }
        .stats-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
            gap: 15px;
            margin-bottom: 20px;
        }
        .stat-item {
            background: rgba(255, 255, 255, 0.03);
            padding: 12px 16px;
            border-radius: 10px;
            border: 1px solid var(--border-color);
        }
        .stat-item .label { font-size: 0.8rem; color: var(--text-sub); }
        .stat-item .value {
            font-size: 1.3rem;
            font-weight: 700;
            color: var(--accent-cyan);
            font-family: 'JetBrains Mono', monospace;
            margin-top: 4px;
        }
        .interactive-panel {
            background: rgba(15, 23, 42, 0.8);
            border: 1px dashed rgba(6, 182, 212, 0.3);
            border-radius: 12px;
            padding: 15px 20px;
            margin-bottom: 20px;
        }
        .interactive-panel h4 { font-size: 0.9rem; color: var(--accent-cyan); margin-bottom: 10px; }
        .slider-group { display: flex; align-items: center; gap: 15px; flex-wrap: wrap; }
        .slider-group label { font-size: 0.85rem; min-width: 120px; }
        .slider-group input[type=range] {
            flex: 1;
            accent-color: var(--accent-cyan);
            cursor: pointer;
        }
        .slider-group .curr-val {
            font-family: 'JetBrains Mono', monospace;
            background: rgba(255, 255, 255, 0.1);
            padding: 2px 8px;
            border-radius: 4px;
            font-size: 0.85rem;
            min-width: 60px;
            text-align: center;
        }
        .img-container {
            text-align: center;
            background: rgba(0, 0, 0, 0.2);
            border-radius: 12px;
            padding: 10px;
            border: 1px solid var(--border-color);
        }
        .img-container img { max-width: 100%; height: auto; border-radius: 8px; display: block; margin: 0 auto; }
    </style>
</head>
<body>
    <div class="container">
        <header>
            <div class="logo-title">IMP302 <span>LAB DASHBOARD</span></div>
            <div class="badge">OUTPUT FILE: IMP302_Report.html</div>
        </header>
        <div id="content-stream">
            <!-- SECTIONS WILL BE INJECTED HERE -->
        </div>
    </div>
</body>
</html>"""
  with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.write(html_template)
  print(f"[OK] Đã khởi tạo khung Dashboard tại: {os.path.abspath(REPORT_FILE)}")


def append_to_report(section_id, section_html):
  """Nối hoặc cập nhật một Section vào file HTML."""
  if not os.path.exists(REPORT_FILE):
    init_dashboard()

  with open(REPORT_FILE, "r", encoding="utf-8") as f:
    content = f.read()

  marker = "<!-- SECTIONS WILL BE INJECTED HERE -->"
  # Bọc section vào thẻ div có id để tránh trùng lặp nếu chạy lại cell
  section_block = f'\n<div id="{section_id}">{section_html}</div>\n{marker}'

  if f'id="{section_id}"' in content:
    # Nếu section đã tồn tại, thay thế khối cũ bằng khối mới
    import re

    pattern = rf'<div id="{section_id}">.*?</div>'
    content = re.sub(
        pattern,
        f'<div id="{section_id}">{section_html}</div>',
        content,
        flags=re.DOTALL,
    )
  else:
    content = content.replace(marker, section_block)

  with open(REPORT_FILE, "w", encoding="utf-8") as f:
    f.write(content)

  print(
      f"[Cập nhật thành công] Đã ghi nội dung '{section_id}' vào"
      f" {REPORT_FILE}."
  )


# Khởi tạo Dashboard
init_dashboard()

[OK] Đã khởi tạo khung Dashboard tại: c:\Users\PC\OneDrive\Desktop\IMP302\IMP302_Report.html


In [7]:
# =====================================================================
# CELL 1: TÍNH TOÁN VÀ XUẤT BÁO CÁO AOD LÊN HTML
# =====================================================================

image_path = r"C:\Users\PC\OneDrive\Desktop\IMP302\images.jpg"
img_bgr = cv2.imread(image_path)
if img_bgr is None:
  raise FileNotFoundError(f"Không tìm thấy ảnh tại: {image_path}")

img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

N, M = img_gray.shape
total_pixels = M * N
aod_pixel = float(np.sum(img_gray.astype(np.float64)) / total_pixels)

# Vẽ biểu đồ Matplotlib
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), facecolor="#1e293b")
for ax in axes:
  ax.set_facecolor("#0f172a")
  ax.tick_params(colors="#94a3b8")
  for spine in ax.spines.values():
    spine.set_color("#334155")

axes[0].imshow(img_rgb)
axes[0].set_title("Original RGB", color="#f8fafc", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(img_gray, cmap="gray")
axes[1].set_title("Grayscale f(n, m)", color="#f8fafc", fontweight="bold")
axes[1].axis("off")

hist, _ = np.histogram(img_gray.flatten(), bins=256, range=[0, 256])
k_vals = np.arange(256)
axes[2].bar(k_vals, hist, width=1.0, color="#38bdf8", alpha=0.7)
axes[2].axvline(
    x=aod_pixel,
    color="#f43f5e",
    linestyle="--",
    linewidth=2,
    label=f"AOD = {aod_pixel:.2f}",
)
axes[2].set_title(
    "Histogram & AOD Center", color="#f8fafc", fontweight="bold"
)
axes[2].set_xlim([0, 255])
axes[2].grid(True, linestyle=":", color="#334155")
legend = axes[2].legend(facecolor="#1e293b", edgecolor="#334155")
for text in legend.get_texts():
  text.set_color("#f8fafc")

plt.tight_layout()
b64_img = fig_to_base64(fig)

# Tạo Section HTML có thanh trượt tương tác
section_aod_html = f"""
<div class="card">
    <div class="card-header">
        <h2>1. Average Optical Density (AOD)</h2>
        <span class="badge">POINT METRIC</span>
    </div>
    <div class="formula-box">
        AOD(f) = (1 / MN) * ΣΣ f(n, m) = (1 / MN) * Σ k·Hf(k)
    </div>
    
    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">Kích thước (M x N)</div>
            <div class="value">{M} x {N}</div>
        </div>
        <div class="stat-item">
            <div class="label">Tổng số điểm ảnh</div>
            <div class="value">{total_pixels:,}</div>
        </div>
        <div class="stat-item">
            <div class="label">Giá trị AOD thực tế</div>
            <div class="value" style="color: #f43f5e;">{aod_pixel:.2f}</div>
        </div>
        <div class="stat-item">
            <div class="label">Dải động (Dynamic Range)</div>
            <div class="value">[{np.min(img_gray)}, {np.max(img_gray)}]</div>
        </div>
    </div>

    <!-- KHỐI TƯƠNG TÁC SLIDER -->
    <div class="interactive-panel">
        <h4>🎮 Khám phá tương tác: Mô phỏng dịch chuyển mức xám ảnh hưởng đến AOD</h4>
        <div class="slider-group">
            <label for="slider_aod">Giả lập độ lệch sáng (Δ):</label>
            <input type="range" id="slider_aod" min="-100" max="100" value="0" 
                   oninput="document.getElementById('val_aod').innerText = this.value;
                            document.getElementById('calc_aod').innerText = (parseFloat({aod_pixel}) + parseFloat(this.value)).toFixed(2);">
            <span class="curr-val" id="val_aod">0</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">→ AOD ước tính mới: </span>
            <span class="curr-val" style="color: #38bdf8;" id="calc_aod">{aod_pixel:.2f}</span>
        </div>
    </div>

    <div class="img-container">
        <img src="{b64_img}" alt="AOD Histogram Output">
    </div>
</div>
"""

append_to_report("section_aod", section_aod_html)

[Cập nhật thành công] Đã ghi nội dung 'section_aod' vào IMP302_Report.html.


In [8]:
# =====================================================================
# CELL 2: ADDITIVE IMAGE OFFSET & APPEND VÀO REPORT HTML
# =====================================================================

K = 256
target_level = K / 2.0  # 128
L_m = aod_pixel

g_float = img_gray.astype(np.float64) - L_m + target_level
img_offset = np.clip(g_float, 0, 255).astype(np.uint8)
aod_calibrated = float(np.mean(img_offset))

# Vẽ biểu đồ đối chiếu
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#1e293b")
for ax in axes:
  ax.set_facecolor("#0f172a")
  ax.tick_params(colors="#94a3b8")
  for spine in ax.spines.values():
    spine.set_color("#334155")

# Ảnh sau khi offset
axes[0].imshow(img_offset, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(
    f"Ảnh sau khi Offset (AOD: {aod_calibrated:.2f})",
    color="#f8fafc",
    fontweight="bold",
)
axes[0].axis("off")

# Histogram đối chiếu
hist_offset, _ = np.histogram(img_offset.flatten(), bins=256, range=[0, 256])
axes[1].bar(
    k_vals,
    hist,
    width=1.0,
    color="#38bdf8",
    alpha=0.3,
    label="Histogram gốc H_f",
)
axes[1].bar(
    k_vals,
    hist_offset,
    width=1.0,
    color="#10b981",
    alpha=0.6,
    label="Histogram sau Offset H_g",
)
axes[1].axvline(
    x=target_level,
    color="#f59e0b",
    linestyle=":",
    linewidth=2,
    label=f"Tâm mục tiêu K/2 = {target_level:.0f}",
)
axes[1].axvline(
    x=aod_calibrated,
    color="#10b981",
    linestyle="--",
    linewidth=2,
    label=f"AOD mới = {aod_calibrated:.2f}",
)
axes[1].set_title(
    "Đối chiếu dịch chuyển Histogram về K/2",
    color="#f8fafc",
    fontweight="bold",
)
axes[1].set_xlim([0, 255])
axes[1].grid(True, linestyle=":", color="#334155")
legend = axes[1].legend(facecolor="#1e293b", edgecolor="#334155")
for text in legend.get_texts():
  text.set_color("#f8fafc")

plt.tight_layout()
b64_img_offset = fig_to_base64(fig)

section_offset_html = f"""
<div class="card">
    <div class="card-header">
        <h2>2. Additive Image Offset</h2>
        <span class="badge">CALIBRATION</span>
    </div>
    <div class="formula-box">
        g_m(n) = f_m(n) - L_m + K/2 &nbsp;|&nbsp; L_m = AOD(f_m), K = 256, K/2 = 128
    </div>

    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">L_m ban đầu</div>
            <div class="value">{L_m:.2f}</div>
        </div>
        <div class="stat-item">
            <div class="label">Độ bù sáng (Offset)</div>
            <div class="value" style="color: #f59e0b;">{target_level - L_m:+.2f}</div>
        </div>
        <div class="stat-item">
            <div class="label">AOD đạt được</div>
            <div class="value" style="color: #10b981;">{aod_calibrated:.2f}</div>
        </div>
        <div class="stat-item">
            <div class="label">Đích chuẩn hóa (Target)</div>
            <div class="value">128.0</div>
        </div>
    </div>

    <div class="interactive-panel">
        <h4>🎮 Khám phá tương tác: Thử nghiệm bù trừ mức sáng tùy ý</h4>
        <div class="slider-group">
            <label for="slider_k2">Chọn mức cân bằng K_target:</label>
            <input type="range" id="slider_k2" min="50" max="200" value="128" 
                   oninput="document.getElementById('val_k2').innerText = this.value;
                            let shift = this.value - {L_m:.2f};
                            document.getElementById('shift_val').innerText = (shift > 0 ? '+' : '') + shift.toFixed(2);">
            <span class="curr-val" id="val_k2">128</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">→ Độ bù tương ứng: </span>
            <span class="curr-val" style="color: #f59e0b;" id="shift_val">{(128 - L_m):+.2f}</span>
        </div>
    </div>

    <div class="img-container">
        <img src="{b64_img_offset}" alt="Additive Offset Output">
    </div>
</div>
"""

append_to_report("section_offset", section_offset_html)

[Cập nhật thành công] Đã ghi nội dung 'section_offset' vào IMP302_Report.html.


In [9]:
# =====================================================================
# CELL 3: MULTIPLICATIVE SCALING (STRETCH / COMPRESS) VÀ GHI REPORT
# =====================================================================

P_comp = 0.5
P_stre = 1.6


def apply_scaling(img, P):
  return np.clip(np.floor(P * img.astype(np.float64) + 0.5), 0, 255).astype(
      np.uint8
  )


img_c = apply_scaling(img_gray, P_comp)
img_s = apply_scaling(img_gray, P_stre)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), facecolor="#1e293b")
for ax in axes:
  ax.set_facecolor("#0f172a")
  ax.tick_params(colors="#94a3b8")
  for spine in ax.spines.values():
    spine.set_color("#334155")

axes[0].imshow(img_c, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(
    f"Ép lại (P = {P_comp}) | AOD: {np.mean(img_c):.1f}",
    color="#f8fafc",
    fontweight="bold",
)
axes[0].axis("off")

axes[1].imshow(img_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(
    f"Ảnh gốc (P = 1.0) | AOD: {aod_pixel:.1f}",
    color="#f8fafc",
    fontweight="bold",
)
axes[1].axis("off")

axes[2].imshow(img_s, cmap="gray", vmin=0, vmax=255)
axes[2].set_title(
    f"Giãn ra (P = {P_stre}) | AOD: {np.mean(img_s):.1f}",
    color="#f8fafc",
    fontweight="bold",
)
axes[2].axis("off")

plt.tight_layout()
b64_img_scale = fig_to_base64(fig)

# Tính số lượng pixel bị cháy trắng ở P=1.6
blown_pixels = np.sum(img_s == 255)
blown_percent = (blown_pixels / total_pixels) * 100

section_scale_html = f"""
<div class="card">
    <div class="card-header">
        <h2>3. Multiplicative Image Scaling</h2>
        <span class="badge">CONTRAST SCALING</span>
    </div>
    <div class="formula-box">
        g(n) = INT[P · f(n) + 0.5] &nbsp;|&nbsp; P > 1: Giãn (Stretch) &nbsp;|&nbsp; P < 1: Ép (Compress)
    </div>

    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">Hệ số ép (P)</div>
            <div class="value">{P_comp}</div>
        </div>
        <div class="stat-item">
            <div class="label">Hệ số giãn (P)</div>
            <div class="value">{P_stre}</div>
        </div>
        <div class="stat-item">
            <div class="label">AOD khi ép (Tối đi)</div>
            <div class="value" style="color: #f59e0b;">{np.mean(img_c):.2f}</div>
        </div>
        <div class="stat-item">
            <div class="label">Pixel bị cháy sáng (P={P_stre})</div>
            <div class="value" style="color: #f43f5e;">{blown_percent:.1f}% ({blown_pixels:,})</div>
        </div>
    </div>

    <div class="interactive-panel">
        <h4>🎮 Khám phá tương tác: Dự đoán ngưỡng bắt đầu bị cháy sáng theo P</h4>
        <div class="slider-group">
            <label for="slider_p">Hệ số nhân P:</label>
            <input type="range" id="slider_p" min="0.2" max="2.5" step="0.05" value="1.6" 
                   oninput="document.getElementById('val_p').innerText = this.value;
                            let p = parseFloat(this.value);
                            let thresh = p > 0 ? Math.floor(255 / p) : 255;
                            document.getElementById('clip_thresh').innerText = Math.min(255, thresh);">
            <span class="curr-val" id="val_p">1.60</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">→ Các pixel có mức xám ≥ mức này sẽ bị cháy trắng: </span>
            <span class="curr-val" style="color: #f43f5e;" id="clip_thresh">159</span>
        </div>
    </div>

    <div class="img-container">
        <img src="{b64_img_scale}" alt="Multiplicative Scaling Output">
    </div>
</div>
"""

append_to_report("section_scale", section_scale_html)

[Cập nhật thành công] Đã ghi nội dung 'section_scale' vào IMP302_Report.html.


In [10]:
# =====================================================================
# CELL 5: IMAGE-AVERAGING FOR NOISE REDUCTION VÀ HOÀN TẤT DASHBOARD
# =====================================================================

import os
import webbrowser
import matplotlib.pyplot as plt
import numpy as np

# 1. Thiết lập tham số và sinh chồng ảnh nhiễu Gaussian
np.random.seed(42)
g = img_gray.astype(np.float64)
sigma = 25.0

noisy_stack = [g + np.random.normal(0, sigma, g.shape) for _ in range(64)]

img_avg1 = np.clip(noisy_stack[0], 0, 255).astype(np.uint8)
img_avg4 = np.clip(np.mean(noisy_stack[:4], axis=0), 0, 255).astype(np.uint8)
img_avg16 = np.clip(np.mean(noisy_stack[:16], axis=0), 0, 255).astype(np.uint8)
img_avg64 = np.clip(np.mean(noisy_stack[:64], axis=0), 0, 255).astype(np.uint8)


# 2. Hàm đo lường chất lượng ảnh PSNR
def get_psnr(orig, noisy):
  mse = np.mean((orig.astype(float) - noisy.astype(float)) ** 2)
  return 10 * np.log10((255.0**2) / (mse + 1e-6))


psnr_1 = get_psnr(img_gray, img_avg1)
psnr_4 = get_psnr(img_gray, img_avg4)
psnr_16 = get_psnr(img_gray, img_avg16)
psnr_64 = get_psnr(img_gray, img_avg64)

# 3. Vẽ biểu đồ đối chiếu
fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), facecolor="#1e293b")
for ax in axes:
  ax.axis("off")

axes[0].imshow(img_avg1, cmap="gray")
axes[0].set_title(
    f"n = 1 | PSNR: {psnr_1:.1f} dB", color="#f43f5e", fontweight="bold"
)

axes[1].imshow(img_avg4, cmap="gray")
axes[1].set_title(
    f"n = 4 | PSNR: {psnr_4:.1f} dB", color="#f8fafc", fontweight="bold"
)

axes[2].imshow(img_avg16, cmap="gray")
axes[2].set_title(
    f"n = 16 | PSNR: {psnr_16:.1f} dB", color="#f8fafc", fontweight="bold"
)

axes[3].imshow(img_avg64, cmap="gray")
axes[3].set_title(
    f"n = 64 | PSNR: {psnr_64:.1f} dB", color="#10b981", fontweight="bold"
)

plt.tight_layout()
b64_img_avg = fig_to_base64(fig)

# 4. Tạo khối giao diện HTML kèm thanh trượt tương tác độ lệch chuẩn
section_avg_html = f"""
<div class="card">
    <div class="card-header">
        <h2>5. Multi-Image Averaging for Noise Reduction</h2>
        <span class="badge">DENOISING</span>
    </div>
    <div class="formula-box">
        (1/n) Σ f_m = g + (1/n) Σ q_m ≈ g &nbsp;|&nbsp; Độ lệch chuẩn nhiễu giảm theo tỷ lệ: σ / √n
    </div>

    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">Nhiễu ban đầu (n=1)</div>
            <div class="value" style="color: #f43f5e;">{psnr_1:.2f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Khử với n = 4</div>
            <div class="value">{psnr_4:.2f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Khử với n = 16</div>
            <div class="value">{psnr_16:.2f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Tối ưu (n = 64)</div>
            <div class="value" style="color: #10b981;">{psnr_64:.2f} dB</div>
        </div>
    </div>

    <div class="interactive-panel">
        <h4>🎮 Khám phá tương tác: Tính độ suy giảm phương sai lý thuyết theo số ảnh n</h4>
        <div class="slider-group">
            <label for="slider_n">Số ảnh tích lũy (n):</label>
            <input type="range" id="slider_n" min="1" max="100" value="64" 
                   oninput="document.getElementById('val_n').innerText = this.value;
                            let val = Math.sqrt(this.value);
                            document.getElementById('sigma_red').innerText = val.toFixed(2) + 'x';">
            <span class="curr-val" id="val_n">64</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">→ Mức giảm độ lệch chuẩn nhiễu (√n): </span>
            <span class="curr-val" style="color: #10b981;" id="sigma_red">8.00x</span>
        </div>
    </div>

    <div class="img-container">
        <img src="{b64_img_avg}" alt="Image Averaging Output">
    </div>
</div>
"""

# 5. Ghi vào file báo cáo chung và tự động mở trình duyệt
append_to_report("section_avg", section_avg_html)

full_path = os.path.abspath(REPORT_FILE)
print("\n ĐÃ HOÀN TẤT BÁO CÁO TOÀN DIỆN!")
print(f"File được lưu tại: {full_path}")
webbrowser.open(f"file://{full_path}")

[Cập nhật thành công] Đã ghi nội dung 'section_avg' vào IMP302_Report.html.

 ĐÃ HOÀN TẤT BÁO CÁO TOÀN DIỆN!
File được lưu tại: c:\Users\PC\OneDrive\Desktop\IMP302\IMP302_Report.html


True

In [11]:
# =====================================================================
# CELL 5: IMAGE-AVERAGING FOR NOISE REDUCTION VÀ HOÀN TẤT DASHBOARD
# =====================================================================

import webbrowser

np.random.seed(42)
g = img_gray.astype(np.float64)
sigma = 25.0

# Tạo 64 ảnh nhiễu và tính trung bình
noisy_stack = [g + np.random.normal(0, sigma, g.shape) for _ in range(64)]

img_avg1 = np.clip(noisy_stack[0], 0, 255).astype(np.uint8)
img_avg4 = np.clip(np.mean(noisy_stack[:4], axis=0), 0, 255).astype(np.uint8)
img_avg16 = np.clip(np.mean(noisy_stack[:16], axis=0), 0, 255).astype(np.uint8)
img_avg64 = np.clip(np.mean(noisy_stack[:64], axis=0), 0, 255).astype(np.uint8)


def get_psnr(orig, noisy):
  mse = np.mean((orig.astype(float) - noisy.astype(float)) ** 2)
  return 10 * np.log10((255.0**2) / (mse + 1e-6))


psnr_1 = get_psnr(img_gray, img_avg1)
psnr_4 = get_psnr(img_gray, img_avg4)
psnr_16 = get_psnr(img_gray, img_avg16)
psnr_64 = get_psnr(img_gray, img_avg64)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), facecolor="#1e293b")
for ax in axes:
  ax.axis("off")

axes[0].imshow(img_avg1, cmap="gray")
axes[0].set_title(
    f"n = 1 | PSNR: {psnr_1:.1f} dB", color="#f43f5e", fontweight="bold"
)

axes[1].imshow(img_avg4, cmap="gray")
axes[1].set_title(
    f"n = 4 | PSNR: {psnr_4:.1f} dB", color="#f8fafc", fontweight="bold"
)

axes[2].imshow(img_avg16, cmap="gray")
axes[2].set_title(
    f"n = 16 | PSNR: {psnr_16:.1f} dB", color="#f8fafc", fontweight="bold"
)

axes[3].imshow(img_avg64, cmap="gray")
axes[3].set_title(
    f"n = 64 | PSNR: {psnr_64:.1f} dB", color="#10b981", fontweight="bold"
)

plt.tight_layout()
b64_img_avg = fig_to_base64(fig)

section_avg_html = f"""
<div class="card">
    <div class="card-header">
        <h2>5. Multi-Image Averaging for Noise Reduction</h2>
        <span class="badge">DENOISING</span>
    </div>
    <div class="formula-box">
        (1/n) Σ f_m = g + (1/n) Σ q_m ≈ g &nbsp;|&nbsp; Độ lệch chuẩn nhiễu giảm theo tỷ lệ: σ / √n
    </div>

    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">Nhiễu ban đầu (n=1)</div>
            <div class="value" style="color: #f43f5e;">{psnr_1:.2f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Khử với n = 4</div>
            <div class="value">{psnr_4:.2f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Khử với n = 16</div>
            <div class="value">{psnr_16:.2f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Tối ưu (n = 64)</div>
            <div class="value" style="color: #10b981;">{psnr_64:.2f} dB</div>
        </div>
    </div>

    <div class="interactive-panel">
        <h4>🎮 Khám phá tương tác: Tính độ suy giảm phương sai lý thuyết theo số ảnh n</h4>
        <div class="slider-group">
            <label for="slider_n">Số ảnh tích lũy (n):</label>
            <input type="range" id="slider_n" min="1" max="100" value="64" 
                   oninput="document.getElementById('val_n').innerText = this.value;
                            let val = Math.sqrt(this.value);
                            document.getElementById('sigma_red').innerText = val.toFixed(2) + 'x';">
            <span class="curr-val" id="val_n">64</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">→ Mức giảm độ lệch chuẩn nhiễu (√n): </span>
            <span class="curr-val" style="color: #10b981;" id="sigma_red">8.00x</span>
        </div>
    </div>

    <div class="img-container">
        <img src="{b64_img_avg}" alt="Image Averaging Output">
    </div>
</div>
"""

append_to_report("section_avg", section_avg_html)

# Mở tự động file HTML trên trình duyệt
full_path = os.path.abspath(REPORT_FILE)
print(f"\n ĐÃ HOÀN TẤT BÁO CÁO TOÀN DIỆN!")
print(f"File được lưu tại: {full_path}")
webbrowser.open(f"file://{full_path}")

[Cập nhật thành công] Đã ghi nội dung 'section_avg' vào IMP302_Report.html.

 ĐÃ HOÀN TẤT BÁO CÁO TOÀN DIỆN!
File được lưu tại: c:\Users\PC\OneDrive\Desktop\IMP302\IMP302_Report.html


True

In [12]:
# =====================================================================
# CELL: ARITHMETIC OPERATIONS & IMAGE-AVERAGING NOISE REDUCTION
# Slide 1.1: 
#   - Pointwise Sum: sum(f_m)
#   - Pointwise Product: prod(f_m)
#   - Noise reduction: (1/n) * sum(f_m) = g + (1/n) * sum(q_m) ≈ g
# =====================================================================

import numpy as np
import matplotlib.pyplot as plt
import cv2

# 1. Khởi tạo ảnh gốc sạch g
g = img_gray.astype(np.float64)
H, W = g.shape
total_px = H * W

# 2. Sinh tập hợp n ảnh nhiễu Gaussian với zero-mean (mean = 0, sigma = 25)
np.random.seed(42)
sigma_noise = 25.0
n_max = 64

noisy_images = [g + np.random.normal(0, sigma_noise, g.shape) for _ in range(n_max)]

# 3. Phép toán số học 1: Pointwise Product (minh họa giữa 2 ảnh có mask/scaling)
# Chuẩn hóa về [0, 1] trước khi nhân điểm để tránh tràn giá trị
f1_norm = noisy_images[0] / 255.0
f2_norm = noisy_images[1] / 255.0
pointwise_prod = np.clip((f1_norm * f2_norm) * 255.0, 0, 255).astype(np.uint8)

# 4. Phép toán số học 2: Averaging với các giá trị n = 1, 4, 16, 64
n_list = [1, 4, 16, 64]
avg_results = {}

def calc_metrics(orig, test_img):
    mse = np.mean((orig - test_img.astype(np.float64)) ** 2)
    psnr = 10 * np.log10((255.0 ** 2) / (mse + 1e-10))
    return float(mse), float(psnr)

for n in n_list:
    stack = noisy_images[:n]
    avg_f = np.mean(stack, axis=0)
    avg_uint8 = np.clip(np.round(avg_f), 0, 255).astype(np.uint8)
    mse_val, psnr_val = calc_metrics(g, avg_uint8)
    avg_results[n] = {
        'img': avg_uint8,
        'mse': mse_val,
        'psnr': psnr_val,
        'theory_std': sigma_noise / np.sqrt(n)
    }

# 5. In số liệu kiểm chứng ra console
print("=" * 70)
print("    HIỆU QUẢ KHỬ NHIỄU THEO SỐ LƯỢNG ẢNH TÍCH LŨY (IMAGE AVERAGING)")
print("=" * 70)
print(f"{'Số ảnh (n)':<12} | {'Độ lệch chuẩn (σ/√n)':<22} | {'MSE':<12} | {'PSNR (dB)':<12}")
print("-" * 70)
for n in n_list:
    res = avg_results[n]
    print(f"{n:<12} | {res['theory_std']:<22.2f} | {res['mse']:<12.2f} | {res['psnr']:<12.2f}")
print("=" * 70)

# 6. Vẽ biểu đồ đối chiếu
fig, axes = plt.subplots(2, 3, figsize=(16, 9), facecolor="#1e293b")
for row in axes:
    for ax in row:
        ax.set_facecolor("#0f172a")
        ax.tick_params(colors="#94a3b8")
        for s in ax.spines.values():
            s.set_color("#334155")

# Hàng 1: Minh họa phép toán số học (Gốc, Nhiễu n=1, Pointwise Product)
axes[0, 0].imshow(img_gray, cmap='gray')
axes[0, 0].set_title("1. Ảnh gốc g", color="#f8fafc", fontweight="bold")
axes[0, 0].axis("off")

axes[0, 1].imshow(avg_results[1]['img'], cmap='gray')
axes[0, 1].set_title(f"2. Ảnh đơn bị nhiễu (n=1)\nPSNR: {avg_results[1]['psnr']:.1f} dB", color="#f43f5e", fontweight="bold")
axes[0, 1].axis("off")

axes[0, 2].imshow(pointwise_prod, cmap='gray')
axes[0, 2].set_title("3. Pointwise Product (f1 ⊗ f2)", color="#f8fafc", fontweight="bold")
axes[0, 2].axis("off")

# Hàng 2: Kết quả khử nhiễu qua trung bình cộng
axes[1, 0].imshow(avg_results[4]['img'], cmap='gray')
axes[1, 0].set_title(f"4. Trung bình n = 4 ảnh\nPSNR: {avg_results[4]['psnr']:.1f} dB", color="#f8fafc", fontweight="bold")
axes[1, 0].axis("off")

axes[1, 1].imshow(avg_results[16]['img'], cmap='gray')
axes[1, 1].set_title(f"5. Trung bình n = 16 ảnh\nPSNR: {avg_results[16]['psnr']:.1f} dB", color="#f8fafc", fontweight="bold")
axes[1, 1].axis("off")

axes[1, 2].imshow(avg_results[64]['img'], cmap='gray')
axes[1, 2].set_title(f"6. Trung bình n = 64 ảnh\nPSNR: {avg_results[64]['psnr']:.1f} dB", color="#10b981", fontweight="bold")
axes[1, 2].axis("off")

plt.tight_layout()
b64_img_avg = fig_to_base64(fig)

# 7. Nhúng khối HTML chuyên dụng vào file báo cáo
section_arithmetic_html = f"""
<div class="card">
    <div class="card-header">
        <h2>Arithmetic Operations & Noise Reduction via Averaging</h2>
        <span class="badge">IMAGE ARITHMETIC</span>
    </div>
    <div class="formula-box">
        f_bar = (1/n) Σ [g + q_m] = g + (1/n) Σ q_m ≈ g &nbsp;|&nbsp; σ_n = σ / √n
    </div>

    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">Nhiễu gốc (n = 1)</div>
            <div class="value" style="color: #f43f5e;">{avg_results[1]['psnr']:.1f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">PSNR khi n = 4</div>
            <div class="value">{avg_results[4]['psnr']:.1f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">PSNR khi n = 16</div>
            <div class="value">{avg_results[16]['psnr']:.1f} dB</div>
        </div>
        <div class="stat-item">
            <div class="label">Tối ưu khi n = 64</div>
            <div class="value" style="color: #10b981;">{avg_results[64]['psnr']:.1f} dB</div>
        </div>
    </div>

    <div class="interactive-panel">
        <h4>🎮 Khám phá tương tác: Mô phỏng mức triệt tiêu nhiễu lý thuyết theo số ảnh n</h4>
        <div class="slider-group">
            <label for="slider_noise_n">Số ảnh gom trung bình (n):</label>
            <input type="range" id="slider_noise_n" min="1" max="100" value="64" 
                   oninput="
                        let n = parseInt(this.value);
                        document.getElementById('lbl_n').innerText = n;
                        let redFactor = Math.sqrt(n);
                        document.getElementById('lbl_red').innerText = redFactor.toFixed(2) + 'x';
                        let remSigma = ({sigma_noise} / redFactor).toFixed(2);
                        document.getElementById('lbl_sigma').innerText = remSigma;
                   ">
            <span class="curr-val" id="lbl_n">64</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">→ Độ lệch chuẩn còn lại σ: </span>
            <span class="curr-val" style="color: #38bdf8;" id="lbl_sigma">{(sigma_noise / np.sqrt(64)):.2f}</span>
            <span style="font-size: 0.85rem; color: #94a3b8;">(Giảm <span id="lbl_red" style="color: #10b981;">8.00x</span>)</span>
        </div>
    </div>

    <div class="img-container">
        <img src="{b64_img_avg}" alt="Image Averaging Output">
    </div>
</div>
"""

append_to_report("section_arithmetic_averaging", section_arithmetic_html)

    HIỆU QUẢ KHỬ NHIỄU THEO SỐ LƯỢNG ẢNH TÍCH LŨY (IMAGE AVERAGING)
Số ảnh (n)   | Độ lệch chuẩn (σ/√n)   | MSE          | PSNR (dB)   
----------------------------------------------------------------------
1            | 25.00                  | 490.61       | 21.22       
4            | 12.50                  | 127.27       | 27.08       
16           | 6.25                   | 32.50        | 33.01       
64           | 3.12                   | 8.30         | 38.94       
[Cập nhật thành công] Đã ghi nội dung 'section_arithmetic_averaging' vào IMP302_Report.html.


In [14]:
# =====================================================================
# CELL 6: GEOMETRIC IMAGE OPERATIONS (SPATIAL MAPPING & INTERPOLATION)
# Loại bỏ slider tương tác - Trình bày kết quả phân tích chuẩn tĩnh
# =====================================================================

import cv2
import matplotlib.pyplot as plt
import numpy as np

# 1. Khởi tạo kích thước và ma trận biến đổi tọa độ
H, W = img_gray.shape
center = (W // 2, H // 2)
theta_deg = 30.0      # Góc xoay
scale_factor = 0.9    # Tỷ lệ co dãn

# Ma trận biến đổi Affine 2x3
M_rot = cv2.getRotationMatrix2D(center, theta_deg, scale_factor)

# 2. Thực hiện Spatial Mapping trên toàn bức ảnh
img_rot_nearest = cv2.warpAffine(img_gray, M_rot, (W, H), 
                                 flags=cv2.INTER_NEAREST, 
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=0)
img_rot_bilinear = cv2.warpAffine(img_gray, M_rot, (W, H), 
                                  flags=cv2.INTER_LINEAR, 
                                  borderMode=cv2.BORDER_CONSTANT, borderValue=0)

# 3. So sánh 2 giải thuật Interpolation trên một vùng cụm nhà (ROI)
roi_y, roi_x, roi_size = H // 3, W // 2, 70
roi = img_gray[roi_y:roi_y + roi_size, roi_x:roi_x + roi_size]
zoom_k = 6.0
dim_zoom = (int(roi_size * zoom_k), int(roi_size * zoom_k))

zoom_nearest = cv2.resize(roi, dim_zoom, interpolation=cv2.INTER_NEAREST)
zoom_bilinear = cv2.resize(roi, dim_zoom, interpolation=cv2.INTER_LINEAR)

# 4. Vẽ đồ thị trực quan Matplotlib (Giao diện Dark Mode đồng bộ)
fig, axes = plt.subplots(2, 3, figsize=(16, 9.5), facecolor='#1e293b')
for ax in axes.ravel():
    ax.set_facecolor('#0f172a')
    ax.axis('off')

# Hàng 1: Spatial Mapping trên toàn bức ảnh
axes[0, 0].imshow(img_gray, cmap='gray')
axes[0, 0].set_title("1. Ảnh gốc f(n)", color='#f8fafc', fontweight='bold', fontsize=11)

axes[0, 1].imshow(img_rot_nearest, cmap='gray')
axes[0, 1].set_title(f"2. Mapping: Nearest (θ={theta_deg}°)", color='#f8fafc', fontweight='bold', fontsize=11)

axes[0, 2].imshow(img_rot_bilinear, cmap='gray')
axes[0, 2].set_title(f"3. Mapping: Bilinear (θ={theta_deg}°)", color='#10b981', fontweight='bold', fontsize=11)

# Hàng 2: So sánh bản chất của phép nội suy (Interpolation)
axes[1, 0].imshow(roi, cmap='gray')
axes[1, 0].set_title(f"4. Vùng ROI gốc ({roi_size}x{roi_size} px)", color='#f8fafc', fontweight='bold', fontsize=11)

axes[1, 1].imshow(zoom_nearest, cmap='gray')
axes[1, 1].set_title("5. Zoom 6x: Nearest Neighbor\n(Răng cưa sắc cạnh / Pixelation)", 
                     color='#f43f5e', fontweight='bold', fontsize=11)

axes[1, 2].imshow(zoom_bilinear, cmap='gray')
axes[1, 2].set_title("6. Zoom 6x: Bilinear Interpolation\n(Biên cạnh chuyển tiếp mượt mà)", 
                     color='#38bdf8', fontweight='bold', fontsize=11)

plt.tight_layout()
b64_img_geom = fig_to_base64(fig)

# 5. Đóng gói Section HTML thuần tĩnh, tinh gọn và chuẩn xác
cos_val = np.cos(np.radians(theta_deg))
sin_val = np.sin(np.radians(theta_deg))

section_geom_html = f"""
<div class="card">
    <div class="card-header">
        <h2>6. Geometric Image Operations</h2>
        <span class="badge">SPATIAL MAPPING & INTERPOLATION</span>
    </div>
    <div class="formula-box">
        g(n) = f(n') = f[a(n)] &nbsp;|&nbsp; Bước 1: Ánh xạ tọa độ không gian &nbsp;|&nbsp; Bước 2: Nội suy (Interpolation)
    </div>

    <div class="stats-grid">
        <div class="stat-item">
            <div class="label">Kích thước ảnh (W x H)</div>
            <div class="value">{W} x {H} px</div>
        </div>
        <div class="stat-item">
            <div class="label">Tâm xoay ảnh (Center)</div>
            <div class="value">({center[0]}, {center[1]})</div>
        </div>
        <div class="stat-item">
            <div class="label">Góc xoay Affine (θ)</div>
            <div class="value" style="color: #38bdf8;">{theta_deg}°</div>
        </div>
        <div class="stat-item">
            <div class="label">Hệ số co dãn (Scale)</div>
            <div class="value" style="color: #10b981;">{scale_factor}</div>
        </div>
    </div>

    <!-- BẢNG CHI TIẾT MA TRẬN ÁNH XẠ TOẠ ĐỘ -->
    <div style="background: rgba(15, 23, 42, 0.6); border: 1px solid var(--border-color); border-radius: 10px; padding: 15px; margin-bottom: 20px;">
        <span style="font-size: 0.85rem; color: var(--text-sub); display: block; margin-bottom: 6px;">Ma trận biến đổi không gian Affine (M):</span>
        <code style="font-family: 'JetBrains Mono', monospace; font-size: 0.85rem; color: #cbd5e1;">
            M = [ [{M_rot[0,0]:.4f}, {M_rot[0,1]:.4f}, {M_rot[0,2]:.1f}], 
                  [{M_rot[1,0]:.4f}, {M_rot[1,1]:.4f}, {M_rot[1,2]:.1f}] ]
        </code>
    </div>

    <div class="img-container">
        <img src="{b64_img_geom}" alt="Geometric Operations Output">
    </div>
</div>
"""

# Ghi đè cập nhật vào file báo cáo
append_to_report("section_geom", section_geom_html)

[Cập nhật thành công] Đã ghi nội dung 'section_geom' vào IMP302_Report.html.
